# Artificial Neural Networks (ANN) - Lab Task #02
## Binary Classification using Logistic Regression with Gradio and Flask Interfaces

---
### Academic Information
- **Student Name:** Aleeya Fatima
- **Registration Number:** 24108108
- **Program:** BS Artificial Intelligence (BS AI - 5A)
- **Department:** Department of Robotics & Artificial Intelligence
- **Submitted To:** Mr. Hassaan Mujtaba
- **Institution:** Shaheed Zulfikar Ali Bhutto Institute of Science & Technology (SZABIST), Islamabad
---


## Task 1: Flood Prediction using Logistic Regression
### Objective
Predict whether a flood will occur at a location (`1 = Yes`, `0 = No`) using meteorological and topographical features:
- **Rainfall** (mm)
- **Elevation** (m)
- **Slope**
- **Distance from the river** (m)

### Methodology
1. Load dataset (`flood_dataset_classification.csv`).
2. Select input features and the binary target `occured`.
3. Split data into 75% training and 25% testing subsets.
4. Standardize input features using `StandardScaler` so that features with different scales (e.g. distance in thousands vs. slope) contribute proportionally.
5. Train a `LogisticRegression` classifier.
6. Evaluate accuracy, confusion matrix, and classification report.
7. Build Gradio and Flask interactive interfaces.


In [ ]:
# Task 1: Flood Prediction using Logistic Regression
# Dataset: flood_dataset_classification.csv
# Goal: Predict if a flood will occur (1 = Yes, 0 = No)
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Step 1: Read the dataset
data = pd.read_csv("flood_dataset_classification.csv")
print("Shape of data:", data.shape)
print(data.head())

# Step 2: Select features and target
features = ["Rainfall", "Elevation", "Slope", "distance"]
target = "occured"
X = data[features]
y = data[target]

# Step 3: Divide data into train (75%) and test (25%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=7)

# Step 4: Scale the values (logistic regression works better on scaled data)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Step 5: Build and train the model
clf = LogisticRegression()
clf.fit(X_train, y_train)

# Step 6: Evaluate the model
predictions = clf.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, predictions) * 100, 2), "%")
print("Confusion Matrix:\n", confusion_matrix(y_test, predictions))
print(classification_report(y_test, predictions))

# Step 7: Function that gives the prediction for new values
def flood_check(rainfall, elevation, slope, distance):
    new_data = pd.DataFrame([[rainfall, elevation, slope, distance]], columns=features)
    new_data = scaler.transform(new_data)
    answer = clf.predict(new_data)[0]
    chance = clf.predict_proba(new_data)[0][1] * 100
    if answer == 1:
        return "Flood Likely (" + str(round(chance, 1)) + "% chance)"
    else:
        return "No Flood Expected (" + str(round(chance, 1)) + "% chance)"


### Task 1 Interactive Interfaces (Gradio & Flask)
Run the Gradio block below for an interactive widget inside the notebook, or start the Flask web app.

In [ ]:
# Gradio App for Task 1
import gradio as gr

with gr.Blocks() as flood_app:
    gr.Markdown("# Flood Prediction System")
    gr.Markdown("Enter the values below and click **Check Flood**")

    rainfall = gr.Number(label="Rainfall (mm)", value=120)
    elevation = gr.Number(label="Elevation (m)", value=50)
    slope = gr.Number(label="Slope", value=2.5)
    distance = gr.Number(label="Distance from River", value=150)
    btn = gr.Button("Check Flood")
    output = gr.Textbox(label="Result")
    btn.click(flood_check, inputs=[rainfall, elevation, slope, distance], outputs=output)

flood_app.launch()


In [ ]:
# Flask App for Task 1
from flask import Flask, request, render_template_string

web = Flask(__name__)
page = """
<html>
<body style="font-family: Arial; background-color: #e8f4fc; padding: 30px;">
    <h1>Flood Prediction System</h1>
    <form action="/predict" method="post">
        <p>Rainfall (mm): <input type="text" name="rainfall" required></p>
        <p>Elevation (m): <input type="text" name="elevation" required></p>
        <p>Slope: <input type="text" name="slope" required></p>
        <p>Distance from River: <input type="text" name="distance" required></p>
        <input type="submit" value="Check Flood">
    </form>
    <h2 style="color: darkblue;">{{ result }}</h2>
</body>
</html>
"""

@web.route("/")
def index():
    return render_template_string(page, result="")

@web.route("/predict", methods=["POST"])
def predict():
    rainfall = float(request.form["rainfall"])
    elevation = float(request.form["elevation"])
    slope = float(request.form["slope"])
    distance = float(request.form["distance"])
    result = flood_check(rainfall, elevation, slope, distance)
    return render_template_string(page, result=result)

# Uncomment to run server:
# web.run(port=5004)


---
## Task 2: Climate Impact Prediction using Logistic Regression
### Objective
Predict whether the economic impact of a climate event will be **High (1)** or **Low (0)**.

### Methodology
1. Load dataset (`realistic_climate_change_impacts.csv`).
2. Synthesize binary target: Calculate median of `EconomicImpact_USD`, labelling values above median as `1` (High) and remainder as `0` (Low).
3. Input features: `TemperatureAnomaly_C`, `CO2Level_ppm`, `PopulationAffected`.
4. Split dataset into 75% train and 25% test partitions.
5. Feature standardization via `StandardScaler`.
6. Train `LogisticRegression` and analyze accuracy (~50.5%), confusion matrix, and classification report.
7. Gradio and Flask interface deployments.


In [ ]:
# Task: Climate Impact Prediction using Logistic Regression
# Dataset: realistic_climate_change_impacts.csv
# Goal: Predict if the economic impact is HIGH (1) or LOW (0)
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Step 1: Read the dataset
data = pd.read_csv("realistic_climate_change_impacts.csv")
print("Shape of data:", data.shape)
print(data.head())

# Step 2: Create the target column
# If economic impact is above the middle value (median) -> 1 (High), otherwise 0 (Low)
middle = data["EconomicImpact_USD"].median()
data["HighImpact"] = (data["EconomicImpact_USD"] > middle).astype(int)
print("Median Economic Impact:", middle)

# Step 3: Select features and target
features = ["TemperatureAnomaly_C", "CO2Level_ppm", "PopulationAffected"]
target = "HighImpact"
X = data[features]
y = data[target]

# Step 4: Divide data into train (75%) and test (25%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=7)

# Step 5: Scale the values
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Step 6: Build and train the model
clf = LogisticRegression()
clf.fit(X_train, y_train)

# Step 7: Evaluate the model
predictions = clf.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, predictions) * 100, 2), "%")
print("Confusion Matrix:\n", confusion_matrix(y_test, predictions))
print(classification_report(y_test, predictions))

# Step 8: Function that gives the prediction for new values
def impact_check(temperature, co2, population):
    new_data = pd.DataFrame([[temperature, co2, population]], columns=features)
    new_data = scaler.transform(new_data)
    answer = clf.predict(new_data)[0]
    chance = clf.predict_proba(new_data)[0][1] * 100
    if answer == 1:
        return "High Economic Impact (" + str(round(chance, 1)) + "% chance)"
    else:
        return "Low Economic Impact (" + str(round(chance, 1)) + "% chance of high)"


### Task 2 Interactive Interfaces (Gradio & Flask)

In [ ]:
# Gradio App for Task 2
import gradio as gr

with gr.Blocks() as climate_app:
    gr.Markdown("# Climate Change Impact Predictor")
    gr.Markdown("Enter the values below and click **Check Impact**")

    temperature = gr.Number(label="Temperature Anomaly (°C)", value=2.1)
    co2 = gr.Number(label="CO2 Level (ppm)", value=425)
    population = gr.Number(label="Population Affected", value=1000000)

    btn = gr.Button("Check Impact")
    output = gr.Textbox(label="Result")
    btn.click(impact_check, inputs=[temperature, co2, population], outputs=output)

climate_app.launch()


In [ ]:
# Flask App for Task 2
from flask import Flask, request, render_template_string

web = Flask(__name__)
page = """
<html>
<body style="font-family: Arial; background-color: #eefbe9; padding: 30px;">
    <h1>Climate Change Impact Predictor</h1>
    <form action="/predict" method="post">
        <p>Temperature Anomaly (°C): <input type="text" name="temperature" required></p>
        <p>CO2 Level (ppm): <input type="text" name="co2" required></p>
        <p>Population Affected: <input type="text" name="population" required></p>
        <input type="submit" value="Check Impact">
    </form>
    <h2 style="color: darkgreen;">{{ result }}</h2>
</body>
</html>
"""

@web.route("/")
def index():
    return render_template_string(page, result="")

@web.route("/predict", methods=["POST"])
def predict():
    temperature = float(request.form["temperature"])
    co2 = float(request.form["co2"])
    population = float(request.form["population"])
    result = impact_check(temperature, co2, population)
    return render_template_string(page, result=result)

# Uncomment to run server:
# web.run(port=5005)
